# 02 · Document Grounding

**목표**: 공개 help.sap.com을 근거로 검색, 주입, 답변을 수행하는 관리형 RAG와 streaming을 관찰합니다.

Grounding은 관련 문서 청크를 먼저 찾고 결과를 prompt에 넣어 근거 없는 답변을 줄입니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️  UI 실행</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">UI: SAP Help Grounding 질의</h2>
<p style="margin: 7px 0;">질문을 입력하면 위에서 구성한 help.sap.com Grounding pipeline으로 답합니다. 답변을 LLM-only 챗봇 결과와 비교해 근거 사용 여부를 관찰하세요.</p>
  </div>
</div>


In [ ]:
from pathlib import Path
import sys
import gradio as gr

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
from lab_common.config import load_settings
from lab_common.ai_core import apply_ai_core_environment, orchestration_url
from gen_ai_hub.orchestration_v2.models.llm_model_details import LLMModelDetails
from gen_ai_hub.orchestration_v2.models.config import ModuleConfig, OrchestrationConfig
from gen_ai_hub.orchestration_v2.models.document_grounding import (
    DocumentGroundingFilter, DocumentGroundingConfig, DocumentGroundingPlaceholders, GroundingModuleConfig)
from gen_ai_hub.orchestration_v2.models.template import Template, PromptTemplatingModuleConfig
from gen_ai_hub.orchestration_v2.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration_v2.service import OrchestrationService

def run_grounding_ui(user_query: str) -> str:
    try:
        settings = load_settings()
        apply_ai_core_environment(settings)
        template = Template(template=[
            SystemMessage(content='Answer only from the provided context. Say when context is insufficient.'),
            UserMessage(content='Request: {{?user_query}}\nContext: {{?grounding_response}}'),
        ])
        grounding = GroundingModuleConfig(config=DocumentGroundingConfig(
            filters=[DocumentGroundingFilter(id='SAPHelp', data_repository_type='help.sap.com')],
            placeholders=DocumentGroundingPlaceholders(input=['user_query'], output='grounding_response'),
        ))
        prompt = PromptTemplatingModuleConfig(
            prompt=template, model=LLMModelDetails(name=settings.llm_model, params={'max_tokens': 1000})
        )
        service = OrchestrationService(
            api_url=orchestration_url(settings),
            config=OrchestrationConfig(modules=ModuleConfig(prompt_templating=prompt, grounding=grounding)),
        )
        result = service.run(placeholder_values={'user_query': user_query})
        return result.final_result.choices[0].message.content
    except Exception as error:
        return f'Grounding 호출 실패: {type(error).__name__}: {error}'

grounding_ui = gr.Interface(
    fn=run_grounding_ui,
    inputs=gr.Textbox(label='Question', lines=3),
    outputs=gr.Textbox(label='Grounded answer', lines=10),
    title='SAP Help Grounding Playground',
)
grounding_ui.launch(inline=True, share=False, show_error=True, server_name="127.0.0.1", height=900)


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li>UI에서 한 번 실행해 결과를 확인한다.</li>
<li>결과가 기대와 다르면 환경 설정과 모델/저장소 이름을 먼저 점검한다.</li>
<li>아래 코드는 같은 기능의 구성 요소를 자세히 살펴보는 선택 심화 자료다.</li>
</ul>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 같은 작업 보기</h2>
<p style="margin: 7px 0;">아래 셀은 UI가 사용하는 구성과 동작을 단계별로 확인하는 자료입니다. 기본 실습에서는 실행하지 않아도 됩니다.</p>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. Grounding pipeline 구성</h2>
<p style="margin: 7px 0;">DocumentGroundingFilter는 검색 위치를 정합니다. DocumentGroundingPlaceholders는 질문 input과 검색 결과 output을 template에 연결합니다. grounding_response는 Grounding 모듈이 자동으로 채웁니다.</p>
  </div>
</div>


In [ ]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
from lab_common.config import load_settings
from lab_common.ai_core import api_headers, apply_ai_core_environment, orchestration_url

settings = load_settings()
apply_ai_core_environment(settings)
ORCH = orchestration_url(settings)

from gen_ai_hub.orchestration_v2.models.llm_model_details import LLMModelDetails
from gen_ai_hub.orchestration_v2.models.config import ModuleConfig, OrchestrationConfig
from gen_ai_hub.orchestration_v2.models.document_grounding import (
    DocumentGroundingFilter, DocumentGroundingConfig,
    DocumentGroundingPlaceholders, GroundingModuleConfig)
from gen_ai_hub.orchestration_v2.models.template import Template, PromptTemplatingModuleConfig
from gen_ai_hub.orchestration_v2.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration_v2.models.streaming import GlobalStreamOptions
from gen_ai_hub.orchestration_v2.service import OrchestrationService

ORCH = ORCH
print("=" * 56)
print("Step 4 · 공개 그라운딩 + 스트리밍")
print("=" * 56)

llm = LLMModelDetails(name=settings.llm_model, params={"max_tokens": 1000})
template = Template(template=[
    SystemMessage(content="You are a helpful assistant."),
    UserMessage(content="""Answer the request using the provided context.
        Request: {{{?user_query}}}
        Context: {{{?grounding_response}}}
        """),
])
filters = [DocumentGroundingFilter(id="SAPHelp", data_repository_type="help.sap.com")]
grounding_config = GroundingModuleConfig(config=DocumentGroundingConfig(
    filters=filters,
    placeholders=DocumentGroundingPlaceholders(input=["user_query"], output="grounding_response"),
))
prompt_template = PromptTemplatingModuleConfig(prompt=template, model=llm)
config = OrchestrationConfig(modules=ModuleConfig(
    prompt_templating=prompt_template, grounding=grounding_config))
service = OrchestrationService(api_url=ORCH, config=config)

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. 근거 기반 질의</h2>
<p style="margin: 7px 0;">service.run에는 질문인 user_query만 전달합니다. 내부적으로 검색, context 주입, LLM 답변이 한 번에 수행됩니다.</p>
  </div>
</div>


In [ ]:
# 1) 그라운딩 질의
print("\n[1] 그라운딩 질의: 'What is Joule?'")
resp = service.run(placeholder_values={"user_query": "What is Joule?"})
print("  응답:", resp.final_result.choices[0].message.content[:300])

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. Streaming</h2>
<p style="margin: 7px 0;">run은 완성된 답을 한 번에 받지만 stream은 생성 조각을 순서대로 받습니다. delta.content는 이번 chunk에서 새로 추가된 텍스트입니다.</p>
  </div>
</div>


In [ ]:
# 2) 스트리밍
print("\n[2] 스트리밍: 'What is SAP AI Core?'")
stream_config = OrchestrationConfig(
    modules=ModuleConfig(prompt_templating=prompt_template, grounding=grounding_config),
    stream=GlobalStreamOptions(enabled=True))
chunks = 0
print("  ", end="")
for chunk in service.stream(config=stream_config,
                            placeholder_values={"user_query": "What is SAP AI Core?"}):
    delta = chunk.final_result.choices[0].delta.content
    if delta:
        chunks += 1
        print(delta, end="", flush=True)
print(f"\n  (총 {chunks}개 청크 수신)")

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. Grounding 전후 비교</h2>
<p style="margin: 7px 0;">LLM만 사용한 답과 같은 질문을 비교해 보세요. template에 Context에 없으면 문서에 없음이라고 답하라는 규칙을 추가하면 근거 범위를 더 명확히 제한할 수 있습니다.</p>
  </div>
</div>


In [ ]:
# 3) Step 1 환각 질문 재질의 → 비교
print("\n[3] 환각 비교: 'How does the data masking of the orchestration service work?'")
resp = service.run(placeholder_values={
    "user_query": "How does the data masking of the orchestration service work?"})
print("  응답(그라운딩 있음):", resp.final_result.choices[0].message.content[:400])

print("\n✅ Step 4 완료 — 그라운딩/스트리밍 동작. [3]을 Step 1 [3]과 비교하세요.")